# 03 — Classification: Logistic Regression & KNN
**Dataset:** Credit Card Fraud Detection — të dhëna të parapërpunuara nga `02_preprocessing.ipynb`

Qëllimi: Trajnojmë dhe vlerësojmë dy klasifikues — **Logistic Regression** (linear) dhe **K-Nearest Neighbors** (distance-based) — me hyperparameter tuning për secilin.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

RANDOM_STATE = 42

In [ ]:
X_train       = np.load('../data/processed/X_train.npy')
X_test        = np.load('../data/processed/X_test.npy')
y_train       = np.load('../data/processed/y_train.npy')
y_test        = np.load('../data/processed/y_test.npy')
feature_names = np.load('../data/processed/feature_names.npy', allow_pickle=True)

print("=" * 50)
print("TË DHËNAT E NGARKUARA")
print("=" * 50)
print(f"X_train : {X_train.shape}  (SMOTE-balanced)")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}  | Fraud: {(y_train==1).sum():,}  Legjitime: {(y_train==0).sum():,}")
print(f"y_test  : {y_test.shape}   | Fraud: {(y_test==1).sum():,}   Legjitime: {(y_test==0).sum():,}")
print(f"\nFeatures ({len(feature_names)}): {list(feature_names)}")

## 1. Logistic Regression
Klasifikues linear që modelon probabilitetin e klasës duke përdorur funksionin sigmoid. Hyperparameter kryesor: **C** (inversi i regularizimit) — vlera e vogël = regularizim i fortë, vlera e madhe = regularizim i dobët.

In [ ]:
param_grid_lr = {
    'C'      : [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver' : ['liblinear']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_lr = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    param_grid_lr,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train, y_train)

print("=" * 50)
print("REZULTATET E GRIDSEARCHCV — LOGISTIC REGRESSION")
print("=" * 50)
print(f"Parametrat më të mirë : {grid_lr.best_params_}")
print(f"F1-score më i mirë (CV): {grid_lr.best_score_:.4f}")

In [ ]:
results_lr = pd.DataFrame(grid_lr.cv_results_)
results_lr = results_lr[['param_C', 'param_penalty', 'mean_test_score', 'std_test_score']]
results_lr = results_lr.sort_values('mean_test_score', ascending=False)
results_lr.columns = ['C', 'Penalty', 'F1 mesatar (CV)', 'Std']
print("Top 8 kombinime:")
print(results_lr.head(8).round(4).to_string(index=False))

### 1.2 Vlerësimi i Logistic Regression
Vlerësojmë modelin më të mirë në **test set** me metrika: Accuracy, Precision, Recall, F1, ROC-AUC dhe Confusion Matrix.

In [ ]:
best_lr   = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)
y_prob_lr = best_lr.predict_proba(X_test)[:, 1]

acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)
auc_lr  = roc_auc_score(y_test, y_prob_lr)

print("=" * 50)
print("METRIKAT — LOGISTIC REGRESSION (Test Set)")
print("=" * 50)
print(f"Accuracy  : {acc_lr:.4f}")
print(f"Precision : {prec_lr:.4f}")
print(f"Recall    : {rec_lr:.4f}")
print(f"F1-score  : {f1_lr:.4f}")
print(f"ROC-AUC   : {auc_lr:.4f}")
print(f"\n{classification_report(y_test, y_pred_lr, target_names=['Legjitime', 'Mashtruese'])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Legjitime', 'Mashtruese'],
            yticklabels=['Legjitime', 'Mashtruese'])
axes[0].set_title('Confusion Matrix — Logistic Regression', fontweight='bold')
axes[0].set_ylabel('Aktuale')
axes[0].set_xlabel('Të parashikuara')

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
axes[1].plot(fpr_lr, tpr_lr, color='steelblue', linewidth=2,
             label=f'Logistic Regression (AUC = {auc_lr:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].set_title('ROC Curve — Logistic Regression', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('../images/lr_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. K-Nearest Neighbors (KNN)
Klasifikues distance-based që klasifikon një pikë bazuar në K fqinjët më të afërt. Hyperparametrat kryesorë: **n_neighbors** (K), **metric** (distanca), **weights** (uniform ose distance-weighted).

In [ ]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 11, 15, 21],
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan']
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_knn.fit(X_train, y_train)

print("=" * 50)
print("REZULTATET E GRIDSEARCHCV — KNN")
print("=" * 50)
print(f"Parametrat më të mirë : {grid_knn.best_params_}")
print(f"F1-score më i mirë (CV): {grid_knn.best_score_:.4f}")

In [ ]:
results_knn = pd.DataFrame(grid_knn.cv_results_)
results_knn = results_knn[['param_n_neighbors', 'param_weights', 'param_metric',
                            'mean_test_score', 'std_test_score']]
results_knn = results_knn.sort_values('mean_test_score', ascending=False)
results_knn.columns = ['K', 'Weights', 'Metric', 'F1 mesatar (CV)', 'Std']
print("Top 8 kombinime:")
print(results_knn.head(8).round(4).to_string(index=False))